# 1D J1J2J3 inference: EuclideanGRU (seed 111-555) 

This is part of the work arxiv:2604.24337 [quant-ph] by HL Dao. For the purpose of reproducing the results, the paths to the trained weights have to be adjusted to match the repo structure.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('../../1_hypnqs_torch_poincare/utility_poincare')
from j1j2j3_hyprnn_train_loop import *
numsamples = 10000

GPU Available:  False


In [2]:
def set_cpu_deterministic(seed):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

In [3]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    # We create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped

def define_load_test(wf,  numsamples,path_to_weights, Ee, clipped_e = False):
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    # This line loads the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        test_samples_after = wf.sample(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = J1J2J3_local_energies(wf, N, J1, J2,J3, Bz, numsamples, test_samples_after, Marshall_sign=True)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2J3_local_energies(wf, N, J1, J2, J3, Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [4]:
N=100
syssize = 100
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)
fname=f'EuclideanGRU_results'

## J2=0.0, J3=0.5

Seed 333 didn't converge. 

In [5]:
J2_ = 0.0
J3_ = 0.5
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_00_05= -53.99140745384424

In [6]:
units = 70
seed=111
set_cpu_deterministic(seed)
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-38.24679946899414-0.003000000026077032j), var E = 19.789600372314453
DMRG energy  is -53.9914
Time taken=0.263 hrs


In [14]:
units = 70
seed=222
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (35.50429916381836+0.008999999612569809j), var E = 3.027600049972534
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.518798828125+0.018200000748038292j), var E = 0.4828999936580658
DMRG energy  is -53.9914
Time taken=0.161 hrs


In [15]:
# No convergence
units = 70
seed=333
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (36.37080001831055+0.0007999999797903001j), var E = 1.826799988746643
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (29.597000122070312-0.01140000019222498j), var E = 16.6112003326416
DMRG energy  is -53.9914
Time taken=0.056 hrs


In [16]:
units = 70
seed=444
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (35.690101623535156+0.008100000210106373j), var E = 2.295799970626831
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-53.29650115966797-0.00570000009611249j), var E = 2.5768001079559326
DMRG energy  is -53.9914
Time taken=0.12 hrs


In [25]:
units = 70
seed=555
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = (35.92639923095703-0.0071000000461936j), var E = 2.265399932861328
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.6151008605957-0.003000000026077032j), var E = 0.57669997215271
DMRG energy  is -53.9914
Time taken=0.193 hrs


## J2=0.2, J3=0.2

In [7]:
J2_ = 0.2
J3_ = 0.2
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_02_02=-43.58595579948936

In [8]:
units = 70
seed=111
set_cpu_deterministic(seed)
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.91019821166992+0.006000000052154064j), var E = 0.6018000245094299
DMRG energy  is -43.586
Time taken=0.137 hrs


In [10]:
units = 70
seed=222
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-33.20600128173828-0.006500000134110451j), var E = 1.336899995803833
DMRG energy  is -43.586
Time taken=0.165 hrs


In [11]:
units = 70
seed=333
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.87260055541992-0.003800000064074993j), var E = 0.3481999933719635
DMRG energy  is -43.586
Time taken=0.151 hrs


In [12]:
units = 70
seed=444
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.3129997253418-0.02160000056028366j), var E = 3.908099889755249
DMRG energy  is -43.586
Time taken=0.155 hrs


In [25]:
units = 70
seed=555
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.03900146484375+0.010099999606609344j), var E = 2.4075000286102295
DMRG energy  is -43.586
Time taken=0.15 hrs


## J2=0.2, J3 = 0.5

In [9]:
J2_ = 0.2
J3_ = 0.5
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_02_05=-49.628675088072384

In [10]:
units = 70
seed=111
set_cpu_deterministic(seed)
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-48.77880096435547+0.000699999975040555j), var E = 2.4375
DMRG energy  is -49.6287
Time taken=0.131 hrs


In [37]:
units = 70
seed=222
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-48.955501556396484+0.01510000042617321j), var E = 1.9703999757766724
DMRG energy  is -49.6287
Time taken=0.168 hrs


In [38]:
units = 70
seed=333
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-48.46120071411133-0.004900000058114529j), var E = 3.7077999114990234
DMRG energy  is -49.6287
Time taken=0.165 hrs


In [39]:
units = 70
seed=444
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-48.892601013183594-0.007600000128149986j), var E = 2.481300115585327
DMRG energy  is -49.6287
Time taken=0.161 hrs


In [40]:
units = 70
seed=555
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-48.367000579833984+0.0027000000700354576j), var E = 0.9710999727249146
DMRG energy  is -49.6287
Time taken=0.171 hrs


## J2=0.5, J3=0.2

In [11]:
J2_ = 0.5
J3_= 0.2
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_05_02= -38.54733450537742

In [12]:
units = 70
seed=111
set_cpu_deterministic(seed)
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_05_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.79990005493164+0.0035000001080334187j), var E = 7.706600189208984
DMRG energy  is -38.5473
Time taken=0.128 hrs


In [42]:
units = 70
seed=222
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.22100067138672+0.00570000009611249j), var E = 0.483599990606308
DMRG energy  is -49.6287
Time taken=0.154 hrs


In [43]:
units = 70
seed=333
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.53070068359375-0.004100000020116568j), var E = 1.2289999723434448
DMRG energy  is -49.6287
Time taken=0.145 hrs


In [44]:
units = 70
seed=444
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-12.954500198364258-0.01940000057220459j), var E = 18.08839988708496
DMRG energy  is -49.6287
Time taken=0.152 hrs


In [45]:
units = 70
seed=555
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2={J2_}|J3={J3_}_EuclGRU_70_rmax=None_ns=80_MsTrue_s{seed}_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU, numsamples, e_wl, Ee_02_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.28929901123047-0.006099999882280827j), var E = 0.5922999978065491
DMRG energy  is -49.6287
Time taken=0.159 hrs
